In [16]:
from pathlib import Path
from torchvision import datasets, transforms

import torch
import time
import pybdm
import numpy as np

seed = 42

# For CUDA
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

data_dir = Path("data")

resize_transform = transforms.Compose([
    transforms.Resize((10, 10)),
    transforms.ToTensor()
])

train_data_resized = datasets.MNIST(
    root=data_dir,
    train=True,
    download=True,
    transform=resize_transform,
)


pixels = []
for image, _ in train_data_resized:
    pixels.append(image.view(-1))

pixels = torch.cat(pixels)
mean = pixels.mean().item()
std = pixels.std(correction=0).item()

print(f"Dataset pixel mean: {mean:.6f}")
print(f"Dataset pixel std:  {std:.6f}")

transform = transforms.Compose([
    transforms.Resize((10, 10)),
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,))
])

train_data = datasets.MNIST(
    root=data_dir,
    train=True,
    download=True,
    transform=transform,
)

test_data = datasets.MNIST(
    root=data_dir,
    train=False,
    download=True,
    transform=transform,
)

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Dataset pixel mean: 0.130896
Dataset pixel std:  0.228951


In [17]:
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit

## Splitting training and evaluation data
# Data is stratified in regards to label to ensure equal representation of labels as in Sakabe et al. (2026)
labels = train_data.targets.numpy()

pool_indices, val_indices = train_test_split(
    np.arange(len(train_data)),
    test_size = 10000,
    stratify=labels,
    random_state=seed
)

# Antagelse: samme for alle arkitekturer
splitter = StratifiedShuffleSplit(n_splits = 200, train_size=25000, random_state = seed)

training_splits = [
    pool_indices[train_index]
    for train_index, _ in splitter.split(pool_indices, labels[pool_indices])
]


In [18]:
#Caching block, if needed copy this into the other versions
from torch.utils.data import TensorDataset, Subset

## Function for caching dataset in memory
def cache_dataset(dataset):
    # Transform once per image
    samples = [dataset[i] for i in range(len(dataset))]

    images = torch.stack([image for image, _ in samples])
    labels = torch.tensor(
        [label for _, label in samples],
        dtype=torch.long
    )

    return TensorDataset(images, labels)


In [ ]:
from torch import nn
from torch.nn import functional as F

# Modules from Saintslab
from scr.quantizer import UniformSymmetricQuantizer                                     # &&
from scr.utils_quantization import attach_weight_quantizers, toggle_quantization        # &&
from scr.qbdm import get_bitplanes, measure_complexity as saintslab_measure_complexity  # &&

TRAINING_MODE = "binary"    # "qat" = Saintslab multibit weights; "binary" = Sakabe-style ±1 weights
QAT_BITS = 1                # Saintslab symmetric quantizer supports >= 2 bits
HIDDEN_SIZES = (16, 8)      # Change to (16, 8), (32,16), and (64,32) for the main experiment

def binarize(x):
   
    binary = torch.where(x > 0, 1.0, -1.0)
    clipped = x.clamp(-1, 1)
    return clipped + (binary - clipped).detach()


class BinaryLinear(nn.Linear):

    def forward(self, x):
        return F.linear(x, binarize(self.weight), self.bias)


class SakabeMLP(nn.Module):
    
    def __init__(self, hidden_sizes=(8, 4), mode="binary", qat_bits=4):
        super().__init__()
        h1, h2 = hidden_sizes                                          # &&
        if mode not in ("binary", "qat"):                              # &&
            raise ValueError(f"Unknown mode: {mode}")                  # &&
        Linear = BinaryLinear if mode == "binary" else nn.Linear       # &&
        self.fc1 = Linear(100, h1, bias=False)                         # &&
        self.bn1 = nn.BatchNorm1d(h1, affine=False)
        self.fc2 = Linear(h1, h2, bias=False)                          # &&
        self.bn2 = nn.BatchNorm1d(h2, affine=False)
        self.fc3 = Linear(h2, 10, bias=False)                          # &&
        self.mode = mode                                               # &&
        self.qat_bits = int(qat_bits) if mode == "qat" else 1          # &&

    def forward(self, x):
        x = x.flatten(start_dim=1)
        x = binarize(self.bn1(self.fc1(x)))
        x = binarize(self.bn2(self.fc2(x)))
        return self.fc3(x)


def build_model():    # &&
    model = SakabeMLP(HIDDEN_SIZES, mode=TRAINING_MODE, qat_bits=QAT_BITS).to(device)
    if TRAINING_MODE == "qat":
        if QAT_BITS < 2:
            raise ValueError("Saintslab's UniformSymmetricQuantizer requires QAT_BITS >= 2")
        attach_weight_quantizers(
            model=model,
            exclude_layers=[],
            quantizer=UniformSymmetricQuantizer(bit_width=QAT_BITS),
            enabled=True,
        )
        toggle_quantization(model, enabled=True)
    return model


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model()
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

SakabeMLP(
  (fc1): BinaryLinear(in_features=100, out_features=16, bias=False)
  (bn1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=False, bias=False, track_running_stats=True)
  (fc2): BinaryLinear(in_features=16, out_features=8, bias=False)
  (bn2): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=False, bias=False, track_running_stats=True)
  (fc3): BinaryLinear(in_features=8, out_features=10, bias=False)
)
Trainable parameters: 1808


In [ ]:
from torch.utils.data import DataLoader
from pybdm.partitions import PartitionIgnore
import pandas as pd

# Load data
train_data_cached = cache_dataset(train_data)
test_data_cached = cache_dataset(test_data)


N_MODELS = 50               
MEASURE_EVERY_N_STEPS = 20  # Complexity measurements every 20 steps 
VAL_EVERY_N_STEPS = 20      # Validation loss measurements every 20 steps
QBDM_MAX_WORKERS = 1        # Saintslab implementation 



# Training hyperparameters (from Sakabe et al. (2026))
learning_rate = 0.001
batch_size = 128
patience = 5
max_epochs = 100


val_data = Subset(train_data_cached, val_indices.tolist())
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data_cached, batch_size=batch_size, shuffle=False)
criterion = nn.CrossEntropyLoss()
bdm_estimator = pybdm.BDM(ndim=2, shape=(4, 4), partition=PartitionIgnore) # Assumption: Partition ignore, ignores incomplete blocks

condition = f"qat_{QAT_BITS}bit" if TRAINING_MODE == "qat" else "binary_1bit"               # &&
results_dir = Path("results") / f"mnist_{HIDDEN_SIZES[0]}_{HIDDEN_SIZES[1]}_{condition}"    # &&
results_dir.mkdir(parents=True, exist_ok=True)
run_summaries = []


# Measure BDM and entropy
def measure_complexity(model):   # &&
    entropy_total = 0.0
    with torch.no_grad():
        if model.mode == "qat":
            _, per_plane_complexity, _ = saintslab_measure_complexity(
                model,
                bit_depths=[model.qat_bits],
                max_workers=QBDM_MAX_WORKERS,
                robust=False,
                tiled=False,
                block_size=None,
            )
            bdm_total = float(sum(per_plane_complexity[model.qat_bits]))
            for layer in (model.fc1, model.fc2, model.fc3):
                # Parametrized layer.weight is the effective weight used in forward.
                planes = get_bitplanes(
                    layer.weight.detach(), num_planes=model.qat_bits,
                    robust=False, block_size=None
                )
                for plane in planes:
                    counts = bdm_estimator.decompose_and_count(plane)
                    entropy_total += bdm_estimator.compute_ent(counts)

        elif model.mode == "binary":
            bdm_total = 0.0
            for layer in (model.fc1, model.fc2, model.fc3):
                binary = (layer.weight.detach() > 0).cpu().numpy().astype(np.int8)
                counts = bdm_estimator.decompose_and_count(binary)
                bdm_total += bdm_estimator.compute_bdm(counts)
                entropy_total += bdm_estimator.compute_ent(counts)
        else: 
            raise ValueError("No mode specified for model. Please specify 'binary' or 'qat'.")

    if not np.isfinite(bdm_total) or not np.isfinite(entropy_total):
        raise RuntimeError("Non-finite QuBD/BDM or entropy measurement")
    return float(bdm_total), float(entropy_total)


def evaluate_validation_loss(model):
    was_training = model.training
    model.eval()
    try:
        total_loss = 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                total_loss += criterion(model(images), labels).item() * labels.size(0)
        return total_loss / len(val_loader.dataset)
    finally:
        model.train(was_training)

# Initialize random seed for reproducibility
torch.manual_seed(seed)

for model_index in range(N_MODELS):

    run_seed = seed + model_index
    torch.manual_seed(run_seed)
    np.random.seed(run_seed)
    loader_rng = torch.Generator().manual_seed(run_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(run_seed)

    train_indices = training_splits[model_index]
    train_subset = Subset(train_data_cached, train_indices)
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, generator=loader_rng)
    model = build_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    print(f"\nModel {model_index + 1}/{N_MODELS}")

    history = {
        "train_loss": [], "val_loss": [], "val_loss_epoch_end": [], "bdm": [], "entropy": [],
    }

    best_epoch_val_loss = float("inf")
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0
    stopped_early = False

    for epoch in range(1, max_epochs + 1):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        epoch_start = time.perf_counter()
        model.train()
        train_loss_perstep = []
        bdm_perstep = []
        entropy_perstep = []
        validation_loss_perstep = []
        step = 0

        for step, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            if step % VAL_EVERY_N_STEPS == 0:
                validation_loss_perstep.append(evaluate_validation_loss(model))

            if step % MEASURE_EVERY_N_STEPS == 0:
                bdm_value, entropy_value = measure_complexity(model)
                bdm_perstep.append(bdm_value)
                entropy_perstep.append(entropy_value)

            loss.backward()
            optimizer.step()
            train_loss_perstep.append(loss.item())

        epoch_val_loss = evaluate_validation_loss(model)

        # Average each metric's measurements within this epoch
        history["train_loss"].append(float(np.mean(train_loss_perstep)))
        history["bdm"].append(float(np.mean(bdm_perstep)))
        history["entropy"].append(float(np.mean(entropy_perstep)))
        history["val_loss"].append(float(np.mean(validation_loss_perstep)))
        history["val_loss_epoch_end"].append(float(epoch_val_loss))
        print(
            f"Epoch {epoch:3d}: train = {history['train_loss'][-1]:.4f}, "
            f"val = {epoch_val_loss:.4f}, QuBD/BDM = {history['bdm'][-1]:.2f}, "
            f"entropy = {history['entropy'][-1]:.4f}"
        )
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        print(f"Full epoch time: {time.perf_counter() - epoch_start:.2f}s")

        if epoch_val_loss < best_epoch_val_loss:
            best_epoch_val_loss = epoch_val_loss
            best_epoch = epoch
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                stopped_early = True
                print(f"Early stopping. Best checkpoint: epoch {best_epoch}")
                break


    model.load_state_dict(best_state)
    model.eval()
    
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(dim=1) == labels).sum().item()
    test_accuracy = correct / len(test_data_cached)
    print(f"Test accuracy: {100 * test_accuracy:.2f}%")

    analysis_epochs = len(history["train_loss"]) - epochs_without_improvement
    results_path = results_dir / f"model_{model_index + 1:03d}.pt"
    torch.save({
        "history": history,
        "analysis_epochs": analysis_epochs,
        "test_accuracy": test_accuracy,
        "split_index": model_index,
        "best_epoch": best_epoch,
        "stopped_early": stopped_early,
        "mode": TRAINING_MODE,
        "qat_bits": QAT_BITS if TRAINING_MODE == "qat" else 1,
        "hidden_sizes": HIDDEN_SIZES,
        "measurement_interval_steps": MEASURE_EVERY_N_STEPS,
        "model_state_dict": best_state
    }, results_path)
    print(f"Saved epoch averages and test accuracy to {results_path}")

    epoch_results = pd.DataFrame(history)
    epoch_results.insert(0, "epoch", range(1, len(epoch_results) + 1))
    epoch_results.to_csv(results_dir / f"{model_index + 1:03d}.csv") 

    run_summaries.append({
        "split_index": model_index,
        "test_accuracy": test_accuracy,
        "best_epoch": best_epoch,
        "results_path": str(results_path)
    })

torch.save(run_summaries, results_dir / "summary.pt")
print(f"Completed {N_MODELS} models. Results saved to {results_dir}")



Model 1/200
Epoch   1: train = 2.2100, val = 1.5298, QuBD/BDM = 3366.70, entropy = 11.6439
Full epoch time: 3.31s
Epoch   2: train = 1.4611, val = 1.3855, QuBD/BDM = 3325.24, entropy = 11.6439
Full epoch time: 2.94s
Epoch   3: train = 1.3437, val = 1.2420, QuBD/BDM = 3266.73, entropy = 11.6299
Full epoch time: 2.94s
Epoch   4: train = 1.3172, val = 1.4931, QuBD/BDM = 3237.17, entropy = 11.6239
Full epoch time: 2.94s
Epoch   5: train = 1.3010, val = 1.3784, QuBD/BDM = 3223.93, entropy = 11.6091
Full epoch time: 2.97s
Epoch   6: train = 1.2628, val = 1.3599, QuBD/BDM = 3224.36, entropy = 11.6159
Full epoch time: 2.99s
Epoch   7: train = 1.2577, val = 1.3720, QuBD/BDM = 3239.57, entropy = 11.6179
Full epoch time: 2.98s
Epoch   8: train = 1.2831, val = 1.2680, QuBD/BDM = 3250.12, entropy = 11.6219
Full epoch time: 2.94s
Early stopping. Best checkpoint: epoch 3
Test accuracy: 58.01%
Saved epoch averages and test accuracy to results\mnist_16_8_binary_1bit\model_001.pt

Model 2/200
Epoch   1